In [1]:
from rag.parser.parser import Parser
from rag.utils.file_validator import FileInspector
from pathlib import Path
from fastembed import TextEmbedding
from rag.parser.code_parser import CodeParser
import json
from copy import deepcopy
from typing import Any

# from transformers import AutoTokenizer
# from tokenizers

/home/user/Documents/Project_5/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
# bge-small's real limit -- use the actual tokenizer, not a line-count guess.
# Line count is a bad proxy: a minified 3-line function and a readable
# 3-line function can differ by 10x in actual token count.
MODEL_NAME = "BAAI/bge-small-en-v1.5"

_MODEL = TextEmbedding(model_name=MODEL_NAME, threads=1)
TOKEN_LIMIT = 480  # small safety margin under the real 512 limit

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def count_tokens(text: str) -> int:
    return _MODEL.token_count(text)

In [2]:
root_path = Path("/home/user/Documents/Project_5/backend/experiments/data/code/documents/spreadsheet")

In [8]:
print(count_tokens("The diagram presents the architecture of a chat application. A user interacts with the frontend through a web or mobile interface. The frontend sends chat requests to a backend API, which handles authentication, validation, conversation management, and message processing. The backend retrieves previous messages from a database and sends the current prompt to an AI model or external model provider. The generated response is returned to the backend, stored as part of the conversation history, and then sent back to the frontend for display. Additional components may include file storage, an embedding service, a vector database for document search, and monitoring tools. Arrows show how requests and responses move between the user interface, backend services, databases, and AI provider."))

151


In [2]:
inspector = FileInspector()
codeParser = CodeParser()
parser = Parser()

In [ ]:
from rag.parser.document_parser import DocumentParser
from rag.parser.markdown_parser import MarkDownParser
from rag.chunker.markdown_chunker import MarkdownChunker
from rag.embedder.embedder import Embedder

doc_parser = DocumentParser()
md_parser = MarkDownParser()
em = Embedder()

all_chunks = []

for file_path in root_path.rglob("*"):
    if not file_path.is_file():
        continue

    inspector.set_file_path(file_path)
    meta = inspector.inspect()
    meta["file"] = file_path

    print(meta)

    result = parser.parse_csv_and_spreadsheets(
        file_name=file_path,
        language=file_path.suffix.lower(),
    )

    if result is not None:
        all_chunks.append(result)
        # print(chunks[-1])

            # markdown_text = doc_parser.parse(file_path)
            # md_parser._set_file_path(file_path)
            # parsed = md_parser.parser(doc=markdown_text, file_path=None)
            # print(markdown_text)
            # print(parsed)


# plain_text = re.sub(r"\\[a-z]+\d* ?", "", rtf_content)
# plain_text = re.sub(r"[{}]", "", plain_text)

# print(plain_text)
# markdown_text = doc_parser.parse(
#     Path(
#         "/home/user/Documents/Project_5/backend/experiments/data/code/file.rtf"
#     )
# )
# # print(markdown_text)
# md_c = MarkdownChunker(
#     file_path=Path(
#         "/home/user/Documents/Project_5/backend/experiments/data/code/file.rtf"
#     ),
#     ext_lang_name=parser.extension_to_language_name,
# )
# parsed = md_parser.parser(doc=markdown_text, file_path=None, use_vision_llm=False)
# # print(markdown_text)
# chunks = md_c._merge_rag_chunks(chunks=parsed, count_token=em.count_tokens)

# data = []
# for chunk in chunks:
#     em_data = {"content": chunk, "tokens": em.count_tokens(chunk)}
#     data.append(em_data)
# chunk = parser.parse_csv_and_spreadsheats(file_name=Path("/home/user/Documents/Project_5/backend/experiments/data/code/documents/spreadsheet/test.ods"), language=".ods")
print(all_chunks)

In [4]:
inspector.set_file_path(Path("/home/user/Documents/Project_5/backend/experiments/data/code/go.mod"))
print(inspector.inspect())

{'category': 'code', 'mime': 'inode/x-empty', 'language_alias': 'modula2', 'language_name': 'Modula-2', 'should_parse': True}


In [ ]:
data = []
complete_data = []
language_alias = ["java", "go", "javascript", "typescript", "python"]
has_read = False
for file_path in root_path.rglob("*"):
    chunks = []
    if file_path.is_file():
        inspector.set_file_path(file_path)
        meta = inspector.inspect()
        if meta.get("language_alias") in language_alias:
            # has_read = True
            codeParser._set_file_path(file_path)
            tree = codeParser.parse_ast()
            if tree:
                # print(tree)
                source_bytes = codeParser._get_source_bytes()
                if not source_bytes:
                    continue
                chunks = codeParser.extract_chunks(tree.root_node)
        for chunk in chunks:
            complete_data.append({"file_path": file_path, "chunk": chunk})
            searilizable_chunk = {k: v for k, v in chunk.items() if k != "_node"}
            data.append(
                {
                    "file_path": str(file_path),
                    "file_name": file_path.name,
                    "chunk": chunk,
                    "len": count_tokens(json.dumps(searilizable_chunk)),
                }
            )
print(data)

[{'file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.java', 'file_name': 'palindrome.java', 'chunk': {'_node': <Node type=class_declaration, start_point=(0, 0), end_point=(18, 1)>, 'kind': 'class_summary', 'name': 'Palindrome', 'comment': None, 'content': 'class Palindrome: methods = main', 'start_line': 0, 'end_line': 18}, 'len': 62}, {'file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.java', 'file_name': 'palindrome.java', 'chunk': {'_node': <Node type=method_declaration, start_point=(1, 4), end_point=(17, 5)>, 'kind': 'function', 'name': 'main', 'comment': None, 'parent_class': 'Palindrome', 'parent_function': None, 'content': 'public static void main(String[] args) {\n        int num = 121;\n        int n = num;\n        int reversed = 0;\n\n        while (n > 0) {\n            int digit = n % 10;\n            reversed = reversed * 10 + digit;\n            n = n / 10;\n        }\n\n        if (num == reversed) {\

In [ ]:
sorted_data = sorted(data, key=lambda x: x["len"], reverse=True)

print(sorted_data)

In [9]:
def create_embeding_text(chunk: dict) -> str:
    parts = []

    kind = chunk.get("kind")
    name = chunk.get("name")
    comment = chunk.get("comment")
    content = chunk.get("content")
    parent_class = chunk.get("parent_class")
    parent_function = chunk.get("parent_function")

    parts.append(f"File: {path.name} | Language: {path.suffix}")

    if kind is not None:
        parts.append(f"Kind: {kind}")

    if name is not None:
        parts.append(f"Name: {name}")

    if parent_class is not None:
        parts.append(f"Parent class: {parent_class}")

    if parent_function is not None:
        parts.append(f"Parent function: {parent_function}")

    code_parts = []

    if comment is not None:
        code_parts.append(str(comment))

    if content is not None:
        code_parts.append(str(content))

    if code_parts:
        parts.append("Code: " + "\n".join(code_parts))

    # Use append, not extend
    text_to_embed = " | ".join(parts)
    return text_to_embed

In [ ]:
def split_oversized_chunk(
    chunk: dict, source_bytes: bytes, token_limit: int = 480
) -> list[dict]:
    clean_chunk = {k: v for k, v in chunk.items() if k != "_node"}

    # 1. Base Check: Does it already fit?
    # print(count_tokens(create_embeding_text(clean_chunk)))
    if count_tokens(create_embeding_text(clean_chunk)) <= token_limit:
        return [clean_chunk]

    node = chunk.get("_node")
    sub_chunks = []

    # 2. Try Tree-Sitter Statement-Level Splitting
    if node:
        body = node.child_by_field_name("body") or node
        statements = [c for c in body.children if c.is_named]

        signature = ""
        if node.type in ["function_declaration", "method_declaration"]:
            signature = source_bytes[node.start_byte : body.start_byte].decode("utf-8")

        current_group = []
        part = 1

        def create_ts_candidate(group, p):
            start = group[0].start_byte
            end = group[-1].end_byte
            text = signature + source_bytes[start:end].decode("utf-8")
            if signature:
                text += "\n}"
            return {
                **clean_chunk,
                "content": text,
                "name": f"{clean_chunk.get('name', 'unnamed')} (part {p})",
                "parent_chunk_id": clean_chunk.get("id"),
                "start_line": group[0].start_point[0],
                "end_line": group[-1].end_point[0],
            }

        for stmt in statements:
            test_group = current_group + [stmt]
            candidate = create_ts_candidate(test_group, part)

            if count_tokens(json.dumps(candidate)) > token_limit and current_group:
                sub_chunks.append(create_ts_candidate(current_group, part))
                part += 1
                current_group = [stmt]
            else:
                current_group.append(stmt)

        if current_group:
            sub_chunks.append(create_ts_candidate(current_group, part))

    # 3. Validation & Text Fallback
    # If _node was missing, or Tree-sitter returned a single statement still over the limit,
    # we force a line-by-line split.
    needs_fallback = False
    if not sub_chunks:
        needs_fallback = True
    else:
        for sc in sub_chunks:
            if count_tokens(json.dumps(sc)) > token_limit:
                needs_fallback = True
                break

    if needs_fallback:
        sub_chunks = []
        lines = clean_chunk["content"].split("\n")
        current_lines = []
        part = 1

        for line in lines:
            test_lines = current_lines + [line]
            candidate_text = "\n".join(test_lines)

            candidate = {
                **clean_chunk,
                "content": candidate_text,
                "name": f"{clean_chunk.get('name', 'unnamed')} (part {part})",
                "parent_chunk_id": clean_chunk.get("id"),
            }

            if (
                count_tokens(create_embeding_text(candidate)) > token_limit
                and current_lines
            ):
                # Flush previous lines
                sub_chunks.append(
                    {
                        **clean_chunk,
                        "content": "\n".join(current_lines),
                        "name": f"{clean_chunk.get('name', 'unnamed')} (part {part})",
                        "parent_chunk_id": clean_chunk.get("id"),
                    }
                )
                part += 1
                current_lines = [line]
            else:
                current_lines.append(line)

        if current_lines:
            sub_chunks.append(
                {
                    **clean_chunk,
                    "content": "\n".join(current_lines),
                    "name": f"{clean_chunk.get('name', 'unnamed')} (part {part})",
                    "parent_chunk_id": clean_chunk.get("id"),
                }
            )

    return sub_chunks

In [ ]:
from pathlib import Path

content_to_embed = []

for data in complete_data:
    file_path = data.get("file_path")

    if not file_path:
        continue

    # Convert the path to a Path object if it is stored as a string
    path = Path(file_path)

    if not path.is_file():
        print(f"File not found: {path}")
        continue

    source_bytes = path.read_bytes()

    chunks = split_oversized_chunk(data.get("chunk"), source_bytes)

    for chunk in chunks:
        text_to_embed = create_embeding_text(chunk)
        content_to_embed.append(
            {"content": text_to_embed, "tokens": count_tokens(text_to_embed)}
        )

print(content_to_embed)

In [ ]:
# data = []
# for chunk in content_to_embed:
#     # print(chunk)
#     chunk_str = str(chunk)
#     data.append({"content": chunk_str, "size": count_tokens(chunk_str)})

sorted_data = sorted(content_to_embed, key=lambda x: x["tokens"], reverse=True)
print(sorted_data)

In [ ]:
data_to_embed = [data["content"] for data in sorted_data]
node_embedding = list(_MODEL.embed(data_to_embed, parallel=1))

node_embedding[0]

In [41]:
from IPython.display import display, HTML

In [28]:
import networkx as nx
from pyvis.network import Network
import matplotlib.pyplot as plt
import umap

In [ ]:
reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(node_embedding)

In [ ]:
import numpy as np
import networkx as nx
import umap
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors

# 1. Dimensionality reduction for visualization coordinates
reducer = umap.UMAP(n_components=2, random_state=42)
coords = reducer.fit_transform(node_embedding)

# 2. Build Graph using k-NN instead of a global similarity threshold
K_NEIGHBORS = 3  # Connect each node only to its top 3 closest matches

nn = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric="cosine")
nn.fit(node_embedding)
distances, indices = nn.kneighbors(node_embedding)

G = nx.Graph()
node_labels = [f"file_{i}.txt" for i in range(len(node_embedding))]

for i in range(len(node_labels)):
    G.add_node(i, label=node_labels[i])
    for j in range(1, K_NEIGHBORS + 1):  # Skip index 0 (self-match)
        neighbor_idx = indices[i][j]
        G.add_edge(i, neighbor_idx)

# 3. Build Edge Traces
edge_x, edge_y = [], []
for u, v in G.edges():
    x0, y0 = coords[u]
    x1, y1 = coords[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    line=dict(width=0.6, color="#aaa"),
    hoverinfo="none",
    mode="lines",
)

# 4. Build Node Traces
node_x = [coords[i][0] for i in G.nodes()]
node_y = [coords[i][1] for i in G.nodes()]
node_degrees = [deg for _, deg in G.degree()]
node_hover = [
    f"File: {G.nodes[i]['label']}<br>Connections: {deg}" for i, deg in G.degree()
]

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers",
    hoverinfo="text",
    text=node_hover,
    marker=dict(
        showscale=True,
        colorscale="YlGnBu",
        reversescale=True,
        color=node_degrees,
        size=8,
        colorbar=dict(thickness=15, title=dict(text="Connections")),
    ),
)

# 5. Render Figure
fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        showlegend=False,
        hovermode="closest",
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    ),
)

fig.show(renderer="browser")

In [6]:
from rag.parser.markdown_parser import MarkDownParser
from rag.chunker.markdown_chunker import MarkdownChunker
from rag.embedder.embedder import Embedder
from pathlib import Path

In [7]:
markdown_file = Path(
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.md"
)

In [8]:
emd = Embedder()
md_parser = MarkDownParser()
md_chunker = MarkdownChunker(markdown_file, inspector.extension_to_language_name)
md_parser._set_file_path(markdown_file)

In [ ]:
ast = md_parser.parse_ast()
if ast is not None:
    raw_chunks = md_parser._parse_markdown(ast.root_node)
    chunks = md_chunker._merge_rag_chunks(raw_chunks, emd.count_tokens)
    data = md_chunker.convert_all_to_embeding_text(chunks)
    converted = []
    for d in data:
        tokens = count_tokens(d)
        converted.append({"data": d, "tokens": tokens})
    print(chunks)

In [ ]:
print(converted)

In [6]:
from rag.parser.parser import Parser
from rag.embedder.embedder import Embedder
from pathlib import Path
parser = Parser()
embedder = Embedder()

Fetching 5 files: 100%|██████████| 5/5 [00:14<00:00,  2.81s/it]


In [8]:
parser._set_file_path(file_path=Path("/home/user/Documents/Project_5/backend/experiments/data/code/package.json"))

In [ ]:
print(parser.general_parser(embedder.count_tokens, target=450))